# G1 Unified Student: walk to pick (sequential pipeline, A*-guided)

End-to-end pilot for a single goal-conditioned MLP that consumes proprio + goal + phase one-hot and outputs 29 joint targets. Data comes from a walk to pick FSM chain that runs entirely inside this folder: no dependency on any sibling repo at runtime.

The walk phase follows an 8-connected A* path around the pick table (only obstacle) so demonstrations detour around the table instead of pulling straight through it. The pick expert takes over inside the walk to pick handover envelope.

Pipeline:
1. **Generate** `outputs/walk_pick_astar_sequential.npz` via K sequential episodes (each with `num_envs=1`, since mjlab's physics diverges subtly at batch>=2 and destabilises the walk to pick handover). Robot spawns uniformly in an annulus of radius `[SPAWN_R_MIN=0.2, SPAWN_R_MAX=5.0]` around the table with fully random yaw ([-π, π)); each episode sees a different (distance, angle-around-table, own-heading) triplet. A* replans per episode at spawn.
2. **Train** the student by MSE BC with phase-balanced sampling. The student only sees proprio + relative box pose + phase — it never gets the A* waypoints as input, so it must learn its own approach policy from the demonstrated trajectories (PPO fine-tune later closes the sim-to-real gap).
3. **Sanity-check** held-out BC MSE.

### Python 3.10 gate and headless MuJoCo
Enforces the interpreter version and sets `MUJOCO_GL=egl` so mjlab renders without a display.

In [1]:
import os, sys
assert sys.version_info[:2] == (3, 10), f"Python 3.10 required, got {sys.version_info[:2]}"
os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, os.getcwd())
print(f"cwd={os.getcwd()}")

cwd=/home/labsca/narayane/unitree-g1-humanoid-monolithic-policy


### Pipeline flags and episode budget
`DEBUG_LIGHT` shrinks the run to a 2 min sanity pass; `REUSE_*` short-circuits data gen or BC training. `USE_ASTAR` selects the walk-phase steering source (True = A* around the pick table; False = reactive pure-pursuit straight at the box) and picks the dataset filename accordingly so both baselines can coexist under `outputs/`. `WORKERS` sets the parallel subprocess pool size; each worker still runs `num_envs=1` because mjwarp's physics diverges at batch>=2 and destabilises the walk-to-pick handover, so process isolation is the only safe way to parallelise.
Spawn radii and step cap feed the CLI args of `src.fsm_continuous_sequential`.

In [ ]:
DEBUG_LIGHT     = False   # True: 4 eps + 3 epochs (~2 min end-to-end sanity)
REUSE_DATASET   = False   # regenerate .npz after tightening the filter to success=1/1 only
REUSE_TRAINING  = False   # skip BC if a checkpoint exists under outputs/logs/g1_monolithic/
USE_ASTAR       = True    # walk phase follows an A* path around the pick table (only obstacle)
WORKERS         = 1 if DEBUG_LIGHT else 20   # parallel subprocess pool; each worker still runs num_envs=1 (process isolation sidesteps the mjwarp batch divergence at num_envs>=2) and uses ~350 MiB VRAM, so cap by GPU memory

K_EPISODES      = 4 if DEBUG_LIGHT else 2000  # at WORKERS=20 on A100: ~2000 eps / (20 parallel * ~40 s/ep) ~= 1-1.5 h; serial baseline was ~22 h
BASE_SEED       = 100                          # per-episode seed = BASE_SEED + i
SPAWN_R_MIN     = 0.2                          # min robot spawn radius (m) around the table
SPAWN_R_MAX     = 5.0                          # max robot spawn radius (m); initial yaw uniform in [-π, π)
MAX_STEPS       = 1200                         # per-episode step cap; far spawns need the extra budget to orbit around the table (seed 109 took 846 steps at radius 5 m)

DATASET_NAME = "walk_pick_astar_sequential.npz" if USE_ASTAR else "walk_pick_sequential.npz"
FAILED_DIR   = "outputs/failed_episodes"       # per-ep .npz of fallen rollouts + metadata.json for offline diagnosis

### Generate the walk-pick sequential dataset
Runs one subprocess per episode via `src.fsm_continuous_sequential` (which spawns `src.fsm_continuous_batch` with `num_envs=1`), launched in a pool of `WORKERS` concurrent processes to sidestep the ~150 MB/episode mjlab context leak and the mjwarp batch divergence that destabilises the handover at `num_envs>=2`. With `USE_ASTAR=True` each subprocess plans an 8-connected A* path from the robot spawn to the table edge nearest the box, then the walk expert pure-pursuits the waypoints and brakes with yaw-only control into the handover envelope; without A* the walk expert reactively pursues the box straight through.
Output: the .npz named by `DATASET_NAME` under `outputs/`, with `unified_obs`, `action`, `phase` arrays.

In [ ]:
import subprocess
from pathlib import Path

DATASET_LOCAL = Path(f"outputs/{DATASET_NAME}").resolve()
REPO_ROOT     = Path(os.getcwd()).resolve()

if REUSE_DATASET and DATASET_LOCAL.exists():
    print(f"[data] reusing existing {DATASET_LOCAL}")
else:
    tag = "A*-guided" if USE_ASTAR else "reactive pure-pursuit"
    print(f"[data] running sequential FSM builder ({tag}, {K_EPISODES} eps, workers={WORKERS}, base_seed={BASE_SEED}, r∈[{SPAWN_R_MIN},{SPAWN_R_MAX}] m, yaw∈[-π,π))")
    cmd = [
        sys.executable, "-m", "src.fsm_continuous_sequential",
        "--episodes", str(K_EPISODES),
        "--base-seed", str(BASE_SEED),
        "--max-steps", str(MAX_STEPS),
        "--r-min", str(SPAWN_R_MIN),
        "--r-max", str(SPAWN_R_MAX),
        "--out", str(DATASET_LOCAL),
        "--failed-dir", FAILED_DIR,
        "--workers", str(WORKERS),
    ]
    if USE_ASTAR:
        cmd += ["--astar"]
    r = subprocess.run(cmd, cwd=str(REPO_ROOT), env={**os.environ, "MUJOCO_GL": "egl"})
    if r.returncode != 0:
        raise RuntimeError(f"Sequential FSM builder failed with exit {r.returncode}")
    assert DATASET_LOCAL.exists(), f"builder returned 0 but {DATASET_LOCAL} was not written"

import numpy as np
blob = np.load(str(DATASET_LOCAL))
obs, act, phase = blob["unified_obs"], blob["action"], blob["phase"]
print(f"dataset: {obs.shape[0]} transitions, obs={obs.shape[1]}, act={act.shape[1]}")
for pid in np.unique(phase):
    name = ["walk", "pick", "carry", "place"][int(pid)]
    n_p = int((phase == pid).sum())
    print(f"  phase {int(pid)} ({name}): {n_p} transitions ({n_p / obs.shape[0] * 100:.1f}%)")

### Diagnose fallen episodes
Loads `outputs/failed_episodes/metadata.json` and reports why the FSM chain fails: distribution of walk-vs-pick failures, how many steps the robot survives, and the top-10 most catastrophic runs. Pure numpy/matplotlib so it runs on any laptop without mjlab or CUDA.

In [ ]:
import json
from pathlib import Path
import numpy as np

meta_path = Path(FAILED_DIR) / "metadata.json"
if not meta_path.exists():
    print(f"[diag] no metadata at {meta_path}; skip (run generate cell first)")
else:
    meta = json.loads(meta_path.read_text())
    per_ep = meta["per_episode"]
    failed = [e for e in per_ep if e["dropped"]]
    kept   = [e for e in per_ep if not e["dropped"]]
    print(f"[diag] {len(failed)}/{meta['episodes']} fell ({100*len(failed)/meta['episodes']:.1f}%); {len(kept)} kept")

    # Walk-phase vs pick-phase failure: an ep is a 'walk fall' if it never reached pick.
    walk_falls = [e for e in failed if e["pick_kept"] == 0]
    pick_falls = [e for e in failed if e["pick_kept"] > 0]
    print(f"  fell during walk: {len(walk_falls)} ({100*len(walk_falls)/max(1,len(failed)):.1f}% of falls)")
  
    print(f"  fell during pick: {len(pick_falls)} ({100*len(pick_falls)/max(1,len(failed)):.1f}% of falls)")

    # Distribution of transitions (proxy for survival time before fall).
    if failed:
        t = np.array([e["transitions"] for e in failed])
        print(f"  survival steps (failed eps): min={t.min()}, p25={np.percentile(t,25):.0f}, median={np.median(t):.0f}, p75={np.percentile(t,75):.0f}, max={t.max()}")
        pk = np.array([e["pick_kept"] for e in pick_falls]) if pick_falls else np.array([0])
        print(f"  steps in pick before fall (pick-falls only): min={pk.min()}, median={np.median(pk):.0f}, max={pk.max()}")

    # Top-10 shortest failures (most catastrophic).
    top = sorted(failed, key=lambda e: e["transitions"])[:10]
    print("\n  10 shortest failures (candidates for video inspection):")
    print(f"    {'seed':>6}  {'total':>6}  {'walk':>6}  {'pick':>6}  npz")
    for e in top:
        npz_short = Path(e['failed_npz']).name if e.get('failed_npz') else '-'
        print(f"    {e['seed']:>6d}  {e['transitions']:>6d}  {e['walk_kept']:>6d}  {e['pick_kept']:>6d}  {npz_short}")


### Train the student by behavioural cloning
Calls `train_bc` from `src.bc_train` with phase-balanced sampling (walk transitions outnumber pick).
`REUSE_TRAINING=True` recovers the latest checkpoint via `src.utils.find_latest_checkpoint`.

In [ ]:
from src.bc_train import train_bc, BCConfig

cfg = BCConfig(
    epochs=3 if DEBUG_LIGHT else 2000,
    batch_size=256 if DEBUG_LIGHT else 8192,
    lr=6e-4 if not DEBUG_LIGHT else 3e-4,
    balance_phases=True,
)

if REUSE_TRAINING:
    from src.utils import find_latest_checkpoint
    ck = find_latest_checkpoint("outputs/logs", experiment_hint="g1_monolithic")
    if ck is None:
        print("[train] no previous run found; training from scratch")
        ck = train_bc(DATASET_LOCAL, cfg=cfg)
    else:
        print(f"[train] reusing {ck}")
else:
    ck = train_bc(DATASET_LOCAL, cfg=cfg)

print(f"[train] final checkpoint: {ck}")

### Held-out MSE sanity check
Loads the checkpoint into `MonolithicPolicyStudent` and reports overall + per-phase MSE on the first 8192 transitions.

In [5]:
import torch
from src.student_policy import MonolithicPolicyStudent

device = "cuda:0" if torch.cuda.is_available() else "cpu"
student = MonolithicPolicyStudent.load(ck, device=device)
student.eval()

n_eval = min(8192, obs.shape[0])
with torch.no_grad():
    x = torch.from_numpy(obs[:n_eval]).float().to(device)
    y = torch.from_numpy(act[:n_eval]).float().to(device)
    y_hat = student(x)
    mse_overall = torch.nn.functional.mse_loss(y_hat, y).item()
print(f"held-out BC MSE on {n_eval} transitions: {mse_overall:.5f}")

p = phase[:n_eval]
for pid in np.unique(p):
    name = ["walk", "pick", "carry", "place"][int(pid)]
    mask = p == pid
    if mask.sum() == 0:
        continue
    with torch.no_grad():
        mse_p = torch.nn.functional.mse_loss(y_hat[mask], y[mask]).item()
    print(f"  {name}: {int(mask.sum())} transitions, MSE={mse_p:.5f}")

held-out BC MSE on 8192 transitions: 0.08220
  walk: 4806 transitions, MSE=0.04329
  pick: 3386 transitions, MSE=0.13743


### TensorBoard hint
Reminder command for inspecting the BC training curves logged under `outputs/logs/g1_monolithic`.

In [6]:
# Launch TensorBoard from a terminal: tensorboard --logdir outputs/logs/g1_monolithic